# Phase 11 — Final Retrieval Experiment

This notebook is the compact execution surface for the final retrieval comparison. Implementation details live in `src/`, machine-readable evidence lives in `outputs/`, and the complete interpretation is in [`write_up/phase11_summary.md`](write_up/phase11_summary.md).

The submitted retriever does not fit a neural ranking model: it evaluates fixed BM25, frozen dense embeddings, and structured entity/context/intent signals. The `run_training` entry point below rebuilds those scores, performs validation selection, and evaluates the frozen configurations.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Run this notebook from the repository root.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.incremental_retrieval import (
    EXPERIMENTS,
    PHASE11_DIR,
    build_phase11_artifacts,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

## Run training and evaluation

Set `force=True` to rebuild embeddings, rankings, metrics, ablations, and manifests. Use `force=False` to load the last reproducible run. Validation selects among the eight fixed configurations by NDCG@10; the test split is only reported after selection.

In [2]:
EXPERIMENT_NAMES = {
    "R-B0": "BM25",
    "R-E1": "BM25 + entities",
    "R-E2": "BM25 + entities + context",
    "R-E3": "Dense",
    "R-E4": "Hybrid",
    "R-E5": "Hybrid + entities",
    "R-E6": "Hybrid + entities + context",
    "R-E7": "Hybrid + entities + context + intent",
}

def final_metrics_matrix(metrics_path=PHASE11_DIR / "metrics.csv"):
    """Return the primary temporal validation/test matrix for the eight planned systems."""
    metrics = pd.read_csv(metrics_path)
    primary = metrics.loc[
        metrics["protocol"].eq("temporal")
        & metrics["scheme"].eq("relevance_grade")
        & metrics["threshold"].eq(1)
        & metrics["experiment"].isin(EXPERIMENTS)
    ].copy()

    validation = (
        primary.loc[
            primary["split"].eq("validation"),
            ["experiment", "ndcg@10", "recall@10", "mrr@10"],
        ]
        .rename(columns={
            "ndcg@10": "validation_ndcg@10",
            "recall@10": "validation_recall@10",
            "mrr@10": "validation_mrr@10",
        })
    )
    test = (
        primary.loc[
            primary["split"].eq("test"),
            [
                "experiment", "ndcg@5", "ndcg@10", "ndcg@20",
                "recall@10", "hit_rate@10", "mrr@10", "judged_fraction@10",
            ],
        ]
        .rename(columns={
            "ndcg@5": "test_ndcg@5",
            "ndcg@10": "test_ndcg@10",
            "ndcg@20": "test_ndcg@20",
            "recall@10": "test_recall@10",
            "hit_rate@10": "test_hit_rate@10",
            "mrr@10": "test_mrr@10",
            "judged_fraction@10": "test_judged_fraction@10",
        })
    )
    matrix = validation.merge(test, on="experiment", validate="one_to_one")
    matrix.insert(1, "configuration", matrix["experiment"].map(EXPERIMENT_NAMES))
    order = {experiment: index for index, experiment in enumerate(EXPERIMENTS)}
    return (
        matrix.sort_values("experiment", key=lambda values: values.map(order))
        .reset_index(drop=True)
    )

def run_training(force=True):
    """Rebuild the fixed retrieval experiment and return metadata plus its final matrix.

    The name is kept as the notebook training entry point. Phase 11 performs no encoder
    fine-tuning; it rebuilds frozen signals, ranks the complete corpus, selects on validation,
    and evaluates the selected configurations.
    """
    metrics_path = PHASE11_DIR / "metrics.csv"
    if force or not metrics_path.exists():
        artifacts = build_phase11_artifacts(PHASE11_DIR)
    else:
        artifacts = {
            "output_dir": str(PHASE11_DIR),
            "metrics": str(metrics_path),
            "mode": "loaded_existing_artifacts",
        }
    return artifacts, final_metrics_matrix(metrics_path)

In [3]:
# Change to True when a full reproducible rebuild is required.
FORCE_REBUILD = False
artifacts, metrics_matrix = run_training(force=FORCE_REBUILD)
artifacts

{'output_dir': '/Users/hol/Documents/Task/doctor-search-service/outputs/retrieval/phase11',
 'metrics': '/Users/hol/Documents/Task/doctor-search-service/outputs/retrieval/phase11/metrics.csv',
 'mode': 'loaded_existing_artifacts'}

## Final metrics matrix

Primary temporal results for `relevance_grade >= 1`. The chosen system is the configuration with the highest validation NDCG@10. Judged fraction is shown because the behavior-derived labels cover only a small part of the corpus.

In [4]:
selected_id = metrics_matrix.loc[
    metrics_matrix["validation_ndcg@10"].idxmax(), "experiment"
]
print(f"Validation-selected system: {selected_id} — {EXPERIMENT_NAMES[selected_id]}")
display(
    metrics_matrix.style
    .format({column: "{:.5f}" for column in metrics_matrix.columns if "@" in column})
    .highlight_max(
        subset=[
            "validation_ndcg@10", "test_ndcg@10",
            "test_recall@10", "test_mrr@10",
        ],
        color="#d9ead3",
    )
)

Validation-selected system: R-E7 — Hybrid + entities + context + intent


,experiment,configuration,validation_ndcg@10,validation_recall@10,validation_mrr@10,test_ndcg@5,test_ndcg@10,test_ndcg@20,test_recall@10,test_hit_rate@10,test_mrr@10,test_judged_fraction@10
0,R-B0,BM25,0.02400,0.06254,0.02351,0.00834,0.01021,0.01772,0.01609,0.06364,0.02016,0.02878
1,R-E1,BM25 + entities,0.02899,0.07139,0.02831,0.00816,0.01303,0.01859,0.02359,0.08182,0.02379,0.03094
2,R-E2,BM25 + entities + context,0.02966,0.07028,0.02778,0.00816,0.01477,0.01822,0.03154,0.08182,0.02389,0.03309
3,R-E3,Dense,0.03079,0.07142,0.02940,0.01058,0.01584,0.02476,0.02972,0.07273,0.01735,0.02878
4,R-E4,Hybrid,0.02545,0.06823,0.02351,0.00881,0.01651,0.01731,0.03528,0.08182,0.02304,0.03165
5,R-E5,Hybrid + entities,0.02467,0.05848,0.02502,0.00797,0.01220,0.01705,0.02177,0.07273,0.02221,0.03165
6,R-E6,Hybrid + entities + context,0.02727,0.06622,0.02465,0.00797,0.01105,0.01691,0.02063,0.06364,0.02091,0.03165
7,R-E7,Hybrid + entities + context + intent,0.03146,0.05118,0.03150,0.00537,0.01940,0.01993,0.04801,0.10909,0.02857,0.03237


## Decision

**R-E7 (hybrid + entities + context + intent)** is the validation-selected research baseline. Its test NDCG@10 is 0.01940 versus 0.01651 for plain hybrid, with test Recall@10 increasing from 0.03528 to 0.04801. These are behavior-label results rather than clinician-adjudicated clinical relevance; see the consolidated Phase 11 summary for failure cases and limitations.

# Phase 12 — Connect Intent to Retrieval

Phase 12 freezes Phase 11 R-E6 as the no-intent starting point, tunes only the intent coefficient on temporal validation NDCG@10, and separates the assisted reference label from the Phase 5 runtime classifier. The notebook includes both the classifier-training entry point and the Phase 12 ranker-training step. The full interpretation is in [`write_up/phase12_summary.md`](write_up/phase12_summary.md).

## Train the intent classifier

Phase 12 uses the Phase 5 T1-E4 classifier: frozen multilingual query embeddings plus entity/context features and class-balanced logistic regression. The training entry point below rebuilds its cross-validation experiments and refits the selected classifier on the 362 retained weak-label queries. The checked-in model was trained with scikit-learn 1.5.2, so `RETRAIN_INTENT_CLASSIFIER` is enabled automatically only in a matching environment; otherwise the versioned artifact is loaded and its recorded training results are displayed.

In [ ]:
import sklearn
from src.intent_improved import (
    COMPARISON_PATH as INTENT_COMPARISON_PATH,
    MODEL_PATH as INTENT_MODEL_PATH,
    build_phase5_artifacts,
)

RETRAIN_INTENT_CLASSIFIER = sklearn.__version__ == "1.5.2"
if RETRAIN_INTENT_CLASSIFIER:
    intent_training_artifacts = build_phase5_artifacts()
else:
    if not INTENT_MODEL_PATH.exists():
        raise RuntimeError("The versioned Phase 5 model is missing; retrain it in the scikit-learn 1.5.2 environment.")
    intent_training_artifacts = {"selected_model": INTENT_MODEL_PATH, "reused": True}

intent_training_results = pd.read_csv(INTENT_COMPARISON_PATH).query("experiment == 'T1-E4'")
print(f"Intent model: {INTENT_MODEL_PATH}")
print(f"Runtime scikit-learn: {sklearn.__version__}; retrained now: {RETRAIN_INTENT_CLASSIFIER}")
display(intent_training_results[["protocol", "macro_f1", "weighted_f1", "accuracy"]])

## Train the Phase 12 ranker

There is no neural ranker fitting in Phase 12. Training means rebuilding the R-E6 features, generating classifier intent scores, evaluating the predefined intent-weight grid on temporal validation NDCG@10, and freezing the selected coefficients before test evaluation. This cell runs that complete training and evaluation pipeline rather than merely loading the result tables.

In [ ]:
from src.intent_retrieval import PHASE12_DIR, build_phase12_artifacts

phase12_training = build_phase12_artifacts()
validation_grid = pd.read_csv(PHASE12_DIR / "validation_grid.csv")
selected_validation_rows = (
    validation_grid.sort_values(["intent_source", "ndcg@10", "intent_weight"], ascending=[True, False, True])
    .groupby("intent_source", as_index=False)
    .first()
)
display(pd.DataFrame([phase12_training]))
display(selected_validation_rows[["intent_source", "intent_weight", "ndcg@10", "recall@10", "mrr@10"]])

## Phase 12 results

The table below reports the primary temporal validation and test metrics for the frozen no-intent baseline, assisted reference intent, hard classifier intent, soft classifier intent, and the explicit Phase 11 R-E7 parity configuration.

In [ ]:
def phase12_results():
    metrics = pd.read_csv(PHASE12_DIR / "metrics.csv")
    primary = metrics.loc[
        metrics["protocol"].eq("temporal")
        & metrics["scheme"].eq("relevance_grade")
        & metrics["threshold"].eq(1)
    ]
    matrix = primary.pivot(index="configuration", columns="split", values=["ndcg@10", "recall@10", "mrr@10"])
    matrix.columns = [f"{split}_{metric}" for metric, split in matrix.columns]
    return matrix.reset_index()

phase12_matrix = phase12_results()
display(phase12_matrix.style.format({column: "{:.5f}" for column in phase12_matrix.columns if "@" in column}))
display(pd.read_csv(PHASE12_DIR / "paired_comparisons.csv").query("protocol == 'temporal' and split == 'test'"))

## Phase 12 decision

| Configuration | Validation NDCG@10 | Test NDCG@10 | Test Recall@10 | Test MRR@10 |
| --- | ---: | ---: | ---: | ---: |
| R-E6 without intent | 0.02727 | 0.01105 | 0.02063 | 0.02091 |
| Predicted soft intent | 0.03084 | 0.01115 | 0.02063 | 0.02136 |
| Predicted hard intent | **0.03146** | 0.01851 | 0.04346 | 0.02766 |
| Assisted intent / R-E7 | **0.03146** | **0.01940** | **0.04801** | **0.02857** |

The validation-selected hard predicted-intent ranker improves temporal test NDCG@10 from **0.01105 to 0.01851**, but its paired 95% interval for the change includes zero and only 10 of 110 eligible queries move. The result is therefore **intent-specific, not global**. Assisted intent reaches 0.01940 and reproduces Phase 11 R-E7 exactly; it remains a reference-label diagnostic rather than a deployable input.

# Phase 13 — Slice-Based Evaluation

Phase 13 evaluates the frozen Phase 11/12 systems across language, entity complexity, contextual complexity, top-level intent, and Pharmacotherapy sub-intent slices. It introduces no new tuning. The full definitions, uncertainty-aware interpretation, and limitations are in [`write_up/phase13_summary.md`](write_up/phase13_summary.md).

In [ ]:
from src.slice_evaluation import PHASE13_DIR, build_phase13_artifacts

phase13_artifacts = build_phase13_artifacts()
display(pd.DataFrame([phase13_artifacts]))

## Slice results and hypothesis checks

The compact views below use the primary temporal test judgments. Empty and sparse slices remain visible so missing support is not mistaken for good performance. `BM25 + structure` (R-E2) isolates entity/context structure from dense retrieval when testing the compositional-complexity hypothesis.

In [ ]:
phase13_metrics = pd.read_csv(PHASE13_DIR / "slice_metrics.csv")
phase13_test = phase13_metrics.query("split == 'test'")
for family in ["language", "entity_count", "contextual_complexity", "intent_top_level", "pharmacotherapy_sub_intent"]:
    view = phase13_test[phase13_test["slice_family"].eq(family)]
    table = view.pivot(index=["slice_name", "queries", "eligible_queries", "support_flag"], columns="system_label", values="ndcg@10").reset_index()
    print(f"\n{family}")
    display(table.style.format({column: "{:.5f}" for column in table.columns if "BM25" in str(column) or "Hybrid" in str(column)}, na_rep="—"))

display(pd.read_csv(PHASE13_DIR / "hypothesis_tests.csv"))

## Phase 13 decision

The strong claim that structured decomposition helps progressively more as query complexity increases is **not supported**: the R-E2 minus BM25 test delta is +0.00389 for no contextual constraints, -0.00052 for one, and +0.02509 for multiple constraints, with only 12 eligible high-complexity queries. Predicted intent is directionally useful for available preferred-content-type intents (+0.01418 NDCG@10) and not for other observed intents (-0.00090); its benefit remains intent-specific rather than global. Entity-count analysis is not identifiable because every supplied query has disease and drug metadata.

# Phase 14 — Detailed Failure Analysis

Phase 14 reviews 12 representative temporal-test top-10 misses from the frozen Phase 12 runtime ranker. It distinguishes plausible title-level ranking/corpus gaps from apparent failures caused by sparse, off-topic behavioral positives. No ranking, model weight, or relevance label is changed. The complete discussion is in [`write_up/phase14_summary.md`](write_up/phase14_summary.md).

In [ ]:
from src.final_failure_analysis import PHASE14_DIR, build_phase14_artifacts

phase14_artifacts = build_phase14_artifacts()
phase14_cases = pd.read_csv(PHASE14_DIR / "failure_cases.csv")
display(pd.DataFrame([phase14_artifacts]))
display(phase14_cases[["query_id", "query", "expected_relevant_content", "retrieved_content", "query_decomposition", "why_retrieval_failed", "what_could_improve_it", "assessment_status"]])

## Phase 14 findings

The dominant residual pattern is incomplete conjunction coverage: a candidate matches the requested context template but not the molecule, or matches the entities but not the information need. Ten of 12 reviewed cases have no title that explicitly covers every requested entity, intent, and contextual constraint. Four cases include low-confidence hard-intent errors. Q180 and Q393 show that behavior-derived misses can be misleading: exact-looking clinical matches are unjudged while off-topic documents receive engagement credit. The next priorities are clinician-adjudicated pooled judgments, passage-level indexing, conjunction-aware compatibility, confidence-gated intent, and explicit corpus-gap handling. Counts are purposive-case diagnostics, not prevalence estimates.

# Phase 15 — CPU-Friendly Learned Reranker

Phase 15 executes the remaining optional CPU-friendly extension from the project plan. Frozen dense retrieval was already evaluated in Phase 11, so this experiment trains pointwise and pairwise logistic rerankers on temporal-train behavioral judgments, selects one with temporal-validation NDCG@10, and compares it with the frozen Phase 12 runtime ranker. The complete methodology and interpretation are in [`write_up/phase15_summary.md`](write_up/phase15_summary.md).

In [ ]:
from src.learned_reranker import PHASE15_DIR, build_phase15_artifacts

phase15_artifacts = build_phase15_artifacts()
phase15_metrics = pd.read_csv(PHASE15_DIR / "metrics.csv")
phase15_primary = phase15_metrics.query("scheme == 'relevance_grade' and threshold == 1")
display(pd.DataFrame([phase15_artifacts]))
display(phase15_primary[["split", "configuration", "positive_queries", "ndcg@10", "recall@10", "mrr@10"]])
display(pd.read_csv(PHASE15_DIR / "paired_comparisons.csv"))

## Phase 15 decision

The class-balanced pointwise logistic model with `C=0.01` wins the predefined challenger grid on validation, moving NDCG@10 from 0.03146 to 0.03363. It does not improve temporal-test retrieval: NDCG@10 falls from 0.01851 to 0.01653 and Recall@10 falls from 0.04346 to 0.03153. Both paired intervals include zero, and only 14 of 110 eligible test queries change in either direction. The frozen Phase 12 runtime ranker remains selected.

## Phase 15 extension — Frozen multilingual cross-encoder

This extension reranks the frozen Phase 12 top 20, 50, or 100 titles with `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`. The checkpoint is not fine-tuned on project labels. Candidate depth and the Phase 12/cross-encoder blend are selected on temporal-validation NDCG@10. The consolidated model-selection analysis is in [`write_up/phase16_summary.md`](write_up/phase16_summary.md).

In [ ]:
from src.cross_encoder_reranker import OUTPUT as PHASE15_CROSS_ENCODER_DIR, build_cross_encoder_artifacts

cross_encoder_artifacts = build_cross_encoder_artifacts()
cross_encoder_metrics = pd.read_csv(PHASE15_CROSS_ENCODER_DIR / "metrics.csv")
display(pd.DataFrame([cross_encoder_artifacts]))
display(cross_encoder_metrics.query("scheme == 'relevance_grade' and threshold == 1")[["split", "configuration", "ndcg@10", "recall@10", "mrr@10"]])
display(pd.read_csv(PHASE15_CROSS_ENCODER_DIR / "paired_comparisons.csv"))

The best cross-encoder challenger uses a top-100 pool and 0.25 cross-encoder weight. It does not beat Phase 12: validation NDCG@10 falls from 0.03146 to 0.02723, and descriptive test NDCG@10 falls from 0.01851 to 0.01545. Recall@10 also declines on test. The frozen Phase 12 runtime ranker remains selected.

## Phase 15 extension — Frozen MedCPT cross-encoder

This retry holds the cross-encoder protocol fixed and substitutes `ncbi/MedCPT-Cross-Encoder`, a PubMed-search-trained biomedical reranker. Candidate depth and blend weight are again selected only by temporal-validation NDCG@10. The consolidated model-selection analysis is in [`write_up/phase16_summary.md`](write_up/phase16_summary.md).

In [ ]:
from src.medcpt_reranker import OUTPUT as PHASE15_MEDCPT_DIR, build_medcpt_artifacts

medcpt_artifacts = build_medcpt_artifacts()
medcpt_metrics = pd.read_csv(PHASE15_MEDCPT_DIR / "metrics.csv")
display(pd.DataFrame([medcpt_artifacts]))
display(medcpt_metrics.query("scheme == 'relevance_grade' and threshold == 1")[["split", "configuration", "ndcg@10", "recall@10", "mrr@10"]])
display(pd.read_csv(PHASE15_MEDCPT_DIR / "paired_comparisons.csv"))

Validation selects a top-50 pool and 0.25 MedCPT weight. MedCPT is better than the general multilingual cross-encoder, but it still does not beat Phase 12: validation NDCG@10 is 0.03027 versus 0.03146, and descriptive test NDCG@10 is 0.01755 versus 0.01851. No eligible test query improves, three worsen, and 107 are unchanged. The frozen Phase 12 runtime ranker remains selected.

## Phase 15 extension — S-PubMedBERT dense embeddings

This full-corpus experiment substitutes or blends `pritamdeka/S-PubMedBert-MS-MARCO` only for Phase 12's dense component. BM25, structured compatibility, predicted intent, splits, and metrics remain fixed. Validation may retain weight zero. The consolidated model-selection analysis is in [`write_up/phase16_summary.md`](write_up/phase16_summary.md).

In [ ]:
from src.pubmedbert_embedding_experiment import OUTPUT as PHASE15_PUBMEDBERT_DIR, build_pubmedbert_embedding_artifacts

pubmedbert_artifacts = build_pubmedbert_embedding_artifacts()
pubmedbert_metrics = pd.read_csv(PHASE15_PUBMEDBERT_DIR / "metrics.csv")
display(pd.DataFrame([pubmedbert_artifacts]))
display(pd.read_csv(PHASE15_PUBMEDBERT_DIR / "validation_grid.csv")[["pubmedbert_weight", "ndcg@10", "recall@10", "mrr@10"]])
display(pubmedbert_metrics.query("scheme == 'relevance_grade' and threshold == 1")[["split", "configuration", "ndcg@10", "recall@10", "mrr@10"]])
display(pd.read_csv(PHASE15_PUBMEDBERT_DIR / "paired_comparisons.csv"))

Validation retains PubMedBERT weight zero. Full replacement lowers hybrid validation/test NDCG@10 from 0.03146/0.01851 to 0.02869/0.01488. PubMedBERT dense-only has a higher descriptive test point estimate than multilingual MiniLM (0.02146 versus 0.01584), but reverses strongly on validation (0.01773 versus 0.03079), so the result is not stable enough for selection. Phase 12 remains the preferred runtime ranker.